# T0 — Return

**Facts used** (classical; module T0 of `notes/temporal_first_curriculum.md`,
and X-6 of `notes/cross_domain_connections.md`).

1. A process is periodic with period $T$ iff $x(t+T)=x(t)$ for all $t$. One
   return $x(t_1)=x(0)$ is compatible with non-periodicity; the second return
   at the same interval is the first witness of a period (X-6: "strictly the
   SECOND return certifies periodicity").
2. Rotation by $\alpha = p/q$ on the circle $\mathbb{R}/\mathbb{Z}$ first
   returns at step $q$; an irrational rotation never returns exactly
   (Poincaré; Hardy & Wright ch. XXIII for the approximation orders).
3. A frequency estimated from $n$ periods of a sinusoid is fuzzy by
   $\Delta f \approx 1/(nT)$: the main lobe of the truncated signal's Fourier
   transform narrows as $1/n$ (Fourier uncertainty; X-6's "frequency-comb
   narrowing").
4. Catalog c01 (Tusi couple, al-Tusi 1247): a circle rolling inside a circle
   of twice its radius traces a straight diameter. Catalog c02: the cycloid's
   cusp at the contact point is semicubical, $|y| \sim |x|^{2/3}$. Catalog c24
   (Sós 1958; Świerczkowski 1959): an irrational rotation's orbit is dense
   with at most three gap lengths; a rational one revisits $q$ points forever.

Everything below is computed; the final cell runs the catalog mutants and
requires them to fail.

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

## 1–2. Counting returns

In [ ]:
def first_return(alpha, eps=1e-9, max_steps=10000):
    th = 0.0
    for n in range(1, max_steps + 1):
        th = (th + alpha) % 1.0
        if min(th, 1 - th) < eps:
            return n
    return None

golden = (math.sqrt(5) - 1) / 2
for label, a in [("1/2", 0.5), ("2/5", 0.4), ("3/7", 3 / 7), ("golden", golden)]:
    print(f"alpha = {label:>6}: first exact return at step {first_return(a)}")

# near-returns of the golden rotation happen at Fibonacci steps and never close
th, best = 0.0, []
for n in range(1, 200):
    th = (th + golden) % 1.0
    d = min(th, 1 - th)
    if not best or d < best[-1][1]:
        best.append((n, d))
print("golden: record near-returns (step, distance):", [(n, f"{d:.2e}") for n, d in best[:8]])

In [ ]:
def check(claimed=lambda p, q: q):
    ok = True
    for p, q in [(1, 2), (2, 5), (3, 7), (5, 11)]:
        ok &= first_return(p / q) == claimed(p, q)
    return ok

# the falsifier discipline: the same check with a wrong claim must fail
falsify(check, {"return-at-numerator": lambda: {"claimed": lambda p, q: p}})

One coordinate's return is not the process's return: the damped signal
below returns to its starting *value* at equal intervals, yet its state
$(x, x')$ never returns - each visit has a smaller velocity.

In [ ]:
x = lambda t: math.sin(t) * math.exp(-t / 4)          # damped - not periodic
# zero crossings of x(t) - x(0) with x(0) = 0
ts = [i * 1e-3 for i in range(1, 12000)]
crossings = [t for a, t in zip(ts, ts[1:]) if x(a) * x(t) <= 0 and x(t) != x(a)]
intervals = [b - a for a, b in zip(crossings, crossings[1:])]
print("return times:", [f"{t:.3f}" for t in crossings[:4]])
print("intervals   :", [f"{d:.3f}" for d in intervals[:3]], "(equal - the value is periodic)")
print("state (x, x') at successive returns:", [(round(x(t), 6), round((x(t + 1e-6) - x(t)) / 1e-6, 3)) for t in crossings[:3]])

## 3. The circumference sharpens as $1/n$

In [ ]:
def main_lobe_halfwidth(n_periods, T=1.0, samples_per_period=64):
    # |sum x(t) e^{-2 pi i f t}| for a cosine observed over n periods; find the first zero above f0 = 1/T
    N = n_periods * samples_per_period
    dt = T / samples_per_period
    xs = [math.cos(2 * math.pi * t * dt / T) for t in range(N)]
    f0 = 1.0 / T
    prev = None
    for k in range(1, 4000):
        f = f0 + k * (0.0005 / n_periods)
        amp = abs(sum(x * cmath.exp(-2j * math.pi * f * t * dt) for t, x in enumerate(xs)))
        if prev is not None and amp > prev:
            return f - f0
        prev = amp
    return None

ns = [1, 2, 4, 8]
ws = [main_lobe_halfwidth(n) for n in ns]
for n, w in zip(ns, ws):
    print(f"n = {n} periods: main-lobe half-width {w:.4f}  (1/n = {1 / n:.4f})")
slope = math.log(ws[-1] / ws[0]) / math.log(ns[-1] / ns[0])
print(f"log-log slope of width vs n: {slope:.3f}")
print(termplot.plot_xy([(math.log(n), math.log(w)) for n, w in zip(ns, ws)], width=50, height=10,
                       title="ln(half-width) vs ln(n periods)", xlabel="ln n", ylabel="ln dF"))

In [ ]:
def check(exponent=-1.0):
    return abs(slope - exponent) < 0.05

falsify(check, {"narrows-as-1/sqrt(n)": lambda: {"exponent": -0.5}})

## 4. Two returns that make a line, and a cusp (catalog c01, c02)

In [ ]:
def hypocycloid(R, r, t):
    return ((R - r) * math.cos(t) + r * math.cos((R - r) / r * t),
            (R - r) * math.sin(t) - r * math.sin((R - r) / r * t))

pts = [hypocycloid(2.0, 1.0, 2 * math.pi * k / 400) for k in range(400)]
print("Tusi couple R = 2r: max |y| =", f"{max(abs(y) for _, y in pts):.1e}", " x spans",
      f"[{min(x for x, _ in pts):.3f}, {max(x for x, _ in pts):.3f}]")
pts3 = [hypocycloid(3.0, 1.0, 2 * math.pi * k / 400) for k in range(400)]
print(termplot.plot_xy(pts3, width=44, height=14, title="R = 3r for contrast: a deltoid, not a line"))

cyc = [(t - math.sin(t), 1 - math.cos(t)) for t in (1e-2, 3e-3, 1e-3)]
exps = [math.log(cyc[i][1] / cyc[i + 1][1]) / math.log(cyc[i][0] / cyc[i + 1][0]) for i in range(2)]
print("cycloid cusp: local exponent d ln y / d ln x =", [f"{e:.4f}" for e in exps], " (2/3 =", f"{2 / 3:.4f})")

In [ ]:
svg = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="-2.2 -2.2 4.4 4.4" width="260" height="260">',
       '<circle cx="0" cy="0" r="2" fill="none" stroke="currentColor" stroke-width="0.03"/>']
d = "M " + " L ".join(f"{x:.3f} {-y:.3f}" for x, y in pts3)
svg.append(f'<path d="{d} Z" fill="none" stroke="#B45309" stroke-width="0.04"/>')
d2 = "M " + " L ".join(f"{x:.3f} {-y:.3f}" for x, y in pts[::8])
svg.append(f'<path d="{d2}" fill="none" stroke="#2F4BC7" stroke-width="0.06"/>')
svg.append("</svg>")
show_svg("".join(svg), "Tusi (blue, R=2r) and deltoid (orange, R=3r)")

## Falsifier: the catalog mutants must fail

In [ ]:
for entry in ("c01_tusi_couple", "c02_cycloid_cusp", "c24_three_gap_kronecker"):
    rc, _ = catalog(entry)
    assert rc == 0, entry
    rc, out = catalog(entry, mutant=True)
    mutant_must_fail(entry, rc, out)